# YuvaIntern Machine Learning Engineer Internship
## Task 2: Hands-On Data Lab Implementation
**Student Name:** Neelotpal Ghosh  
**Admission / Roll No:** 24SCSE1650001  
**Organization:** YuvaIntern  
**Course:** Machine Learning Engineer Virtual Internship  

---
### Objectives:
1. Set up data science environment with Python, Jupyter, and essential libraries (`pandas`, `numpy`, `matplotlib`, `seaborn`).
2. Complete practical exercises in data manipulation, visualization, and basic exploratory analysis.

## 1. Environment Verification & Library Setup
Checking version consistency across the core numerical computing and visualization stack.

In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print(f"Python Version    : {sys.version.split()[0]}")
print(f"NumPy Version     : {np.__version__}")
print(f"Pandas Version    : {pd.__version__}")
print(f"Seaborn Version   : {sns.__version__}")

sns.set_theme(style="whitegrid")

## 2. Ingestion & Structured Data Manipulation
Loading the generated multi-feature workforce dataset (`workforce_lab_data.csv`), inspecting missing values, executing group-wise imputation, and computing summary statistics.

In [2]:
df = pd.read_csv("workforce_lab_data.csv")
print("Shape:", df.shape)
print("\nFirst 5 records:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Imputation using group median
df["YearsExperience"] = df.groupby("Department")["YearsExperience"].transform(lambda x: x.fillna(x.median()))
df["PerformanceScore"] = df.groupby("Department")["PerformanceScore"].transform(lambda x: x.fillna(x.median()))
print("\nVerification post-imputation null count:", df.isnull().sum().sum())

### 2.2 Feature Engineering & Multi-Level Aggregation

In [3]:
# Derived metrics
df["SalaryPerYearExp"] = np.round(df["AnnualSalary"] / np.maximum(df["YearsExperience", 1.0]), 2)
df["PerformanceEfficiency"] = np.round(df["PerformanceScore"] / df["WeeklyHours"], 3)

# Department summary
dept_summary = df.groupby("Department").agg(
    TotalEmployees=("EmployeeID", "count"),
    MeanSalary=("AnnualSalary", "mean"),
    MedianSalary=("AnnualSalary", "median"),
    MeanExperience=("YearsExperience", "mean"),
    MeanPerformance=("PerformanceScore", "mean")
).round(2).sort_values("MeanSalary", ascending=False)

display(dept_summary)

### 2.3 Reshaping with Pivot Tables

In [4]:
pivot_table = df.pivot_table(
    values="AnnualSalary",
    index="Department",
    columns="EducationLevel",
    aggfunc="mean"
).round(2)

display(pivot_table)

## 3. Statistical Data Visualizations
Creating descriptive charts using Matplotlib and Seaborn.

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution plot
sns.histplot(df["AnnualSalary"], kde=True, color="#102C57", ax=axes[0])
axes[0].set_title("Annual Salary Distribution (KDE)")

# Boxplot by Department and Education
sns.boxplot(data=df, x="Department", y="AnnualSalary", hue="EducationLevel", ax=axes[1])
axes[1].set_title("Salary by Department & Education Level")
axes[1].tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.show()

### 3.2 Correlation Analysis

In [6]:
plt.figure(figsize=(8, 6))
numeric_cols = ["Age", "YearsExperience", "AnnualSalary", "PerformanceScore", "WeeklyHours"]
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix of Workforce Features")
plt.show()